#Step 3: Standard Training Pipelines for DL Models

  # Research/Learn

 **Differences between ML and DL pipelines**

  * Definition:
    * ML is a subset of Ai that enables systems to learn from data using algorithms without explicit programming.
    * DL is a specialized subset of ML that utilizes multilayterd neural networks to model complex patterns in data.
  * Data requirements:
    * ML perform will with smaller, strutured datasets. Model can be saturated because, at a certain point, the model can no longer learn.
    * DL perform with with big datasets. Because it will adjust weight to adapt training dataset.
  * Training Time
      * ML generally takes less time to train, ranging from a few seconds to a few hours, because the algorithms and mathematical operations are relatively simpler.
      * DL typically requires significantly more time to train, often taking hours, days, or even weeks due to the massive number of parameters in deep neural networks and the large volume of data.
  * Computational Power
    * ML models are less resource-intensive and can usually be trained on standard CPUs .
    * DL models demand immense computational resources, requiring specialized hardware accelerators like GPUs or TPUs  to handle millions of matrix operations in parallel.

  **Computer vision models:** Hugging Face hosts a vast ecosystem of computer vision (CV) models, ranging from traditional Convolutional Neural Networks (CNNs) to modern Vision Transformers (ViTs) and multimodal Vision-Language Models (VLMs). These models can be used for tasks like image classification, object detection, segmentation, and image generation, primarily through the transformers library

**Advanced NLP models in HF:** Hugging Face (HF) hosts a vast repository of state-of-the-art NLP models that leverage the Transformer architecture for superior performance in text understanding and generation. These models are readily accessible through the HF Transformers library and can be used for tasks like text classification, named entity recognition (NER), and translation.

Feature extractors are components used to transform raw data (such as images or text) into numerical representations (features) that machine learning or deep learning models can process.

*   In traditional ML: feature extraction is often manual (e.g., SIFT, HOG for images; TF-IDF for text).
*   In DL (especially with Transformers): feature extraction is mostly automated and learned by the model.
*   In Hugging Face: feature extractors (or processors) handle tasks like resizing images, normalization, tokenization, and converting inputs into tensors.

**Image/text processing:** Image and text processing refer to the preprocessing steps applied to raw data before feeding it into models. These steps ensure that the data is in the correct format and improves model performance.

* Image processing: resizing, normalization, augmentation (flip, crop, etc.)
* Text processing: tokenization, padding, truncation, removing noise
   In Hugging Face:
  * Text → handled by tokenizers
   * Images → handled by image processors / feature extractors


#Action: Train a DL model using the same high-level HF tools.
Summary action, for what i do in this action. First, I load dataset(emotion). Second, I loading distilbert-base-uncased Tokenizer and Model. I define a preprocessing function for training. Third, I use Trainer for training model. Last I push to HF_Hub.



In [ ]:
from huggingface_hub import notebook_login

notebook_login()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
# 1. Setup
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments
)
from datasets import load_dataset
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

# 2. Load dataset
dataset = load_dataset("emotion")  #

# 3. Load model & tokenizer
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=6  # 6 classes
)

# 4. Preprocessing
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",  #
        truncation=True,
        max_length=128
    )

tokenized_datasets = dataset.map(tokenize_function, batched=True)

# 5. Training arguments (DL-specific settings)
training_args = TrainingArguments(
    output_dir="./results_dl",
    eval_strategy="epoch",
    learning_rate=2e-5,           # Typical cho fine-tuning BERT
    per_device_train_batch_size=256,
    per_device_eval_batch_size=256,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_steps=500,
    logging_dir='./logs',
    logging_steps=10,
    save_strategy="epoch",
    load_best_model_at_end=True,
)

# 6. Metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, predictions),
        'f1': f1_score(labels, predictions, average='weighted')
    }

# 7. Train
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    compute_metrics=compute_metrics,
)

trainer.train()

# 8. Evaluate
eval_results = trainer.evaluate()
print(eval_results)



Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,1.707366,1.674282,0.384500,0.266238
2,1.476849,1.371001,0.561000,0.437249
3,1.046241,0.912244,0.699500,0.632474


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


{'eval_loss': 0.9122439026832581, 'eval_accuracy': 0.6995, 'eval_f1': 0.6324735939603796, 'eval_runtime': 7.0976, 'eval_samples_per_second': 281.785, 'eval_steps_per_second': 1.127, 'epoch': 3.0}


In [ ]:

trainer.args.num_train_epochs = 5


trainer.train(resume_from_checkpoint=True)


eval_results_after_more_training = trainer.evaluate()
print(eval_results_after_more_training)

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


Epoch,Training Loss,Validation Loss,Accuracy,F1
4,0.528007,0.473626,0.864000,0.850621
5,0.305989,0.260184,0.920000,0.919318


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


{'eval_loss': 0.2601839601993561, 'eval_accuracy': 0.92, 'eval_f1': 0.9193175411087859, 'eval_runtime': 7.2207, 'eval_samples_per_second': 276.981, 'eval_steps_per_second': 1.108, 'epoch': 5.0}


In [ ]:
# 1.update epoch i want
trainer.args.num_train_epochs = 10

# 2. start training from the last checkpoint
trainer.train(resume_from_checkpoint=True)

# 3. Reevaluate this model
eval_results_after_more_training = trainer.evaluate()
print(eval_results_after_more_training)

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


Epoch,Training Loss,Validation Loss,Accuracy,F1
6,0.201348,0.193038,0.930500,0.930552
7,0.149950,0.173414,0.932000,0.931614
8,0.115550,0.168442,0.935000,0.934565
9,0.100429,0.151046,0.935000,0.934500
10,0.087342,0.148078,0.935500,0.935282


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


{'eval_loss': 0.14807820320129395, 'eval_accuracy': 0.9355, 'eval_f1': 0.9352824128457298, 'eval_runtime': 6.8746, 'eval_samples_per_second': 290.924, 'eval_steps_per_second': 1.164, 'epoch': 10.0}


In [ ]:
# 9. Push to Hub
repo_name = "duyb2207513/distilbert-emotion"

# Push model and tokenizer to HF_Hub
model.push_to_hub(repo_name)
tokenizer.push_to_hub(repo_name)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...nmthoqo/model.safetensors:   0%|          |  574kB /  268MB            

README.md: 0.00B [00:00, ?B/s]

CommitInfo(commit_url='https://huggingface.co/duyb2207513/distilbert-emotion/commit/f4907d354f5af67b251efa8ed912f7e18963178f', commit_message='Upload tokenizer', commit_description='', oid='f4907d354f5af67b251efa8ed912f7e18963178f', pr_url=None, repo_url=RepoUrl('https://huggingface.co/duyb2207513/distilbert-emotion', endpoint='https://huggingface.co', repo_type='model', repo_id='duyb2207513/distilbert-emotion'), pr_revision=None, pr_num=None)

# Comparation Documents Machine Learning Step and Deep Learning Step For text classification:

## Traditional ML Pipeline
* Load and preprocess dataset
* Apply manual feature engineering
* Convert data into numerical features (e.g., TF-IDF)
* Initialize machine learning model
* Train model using standard .fit() workflow
* Evaluate model performance
* Save or deploy trained model
## Deep Learning Pipeline
* Load and preprocess dataset
* Use tokenizer
* Load pretrained deep learning model
* Configure training arguments and training pipeline
* Fine-tune/train model using Trainer API or training loop
* Evaluate model performance
* Save and push trained model to Hugging Face Hub
## Differences
* ML requires manual feature engineering while DL automatically learns representations
* ML models are usually lightweight and CPU-friendly
* DL models require more computation and often use GPU acceleration

